# Fehler-Fallstudien: IPA-Tokenisierungsprobleme

Dieses Notebook zeigt konkrete Beispiele, bei denen das Wort-Level-Mapping versagt —
z.B. wenn ein HD-Kompositum vom Whisper-Modell inkonsistent in mehrere IPA-Tokens aufgespalten wird.

In [ ]:
import re
from collections import Counter
from pathlib import Path

import pandas as pd
from IPython.display import display, HTML

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "Data"

df = pd.read_csv(DATA_DIR / "transcriptions_tenses.csv")
df_ost = df[df["dialect_region"] == "Ostschweiz"].copy().reset_index(drop=True)
print(f"Sätze (Ostschweiz): {len(df_ost):,}")

In [ ]:
def show_word_cases(keyword, df_source=df_ost):
    """Zeigt alle Sätze mit einem bestimmten HD-Wort als formatierte HTML-Tabelle."""
    mask = df_source["sentence"].str.contains(
        rf"\b{re.escape(keyword)}\b", case=False, regex=True
    )
    df_kw = df_source[mask][["sentence", "ipa_audio", "ipa_reference"]].reset_index(drop=True)

    def hl_hd(text):
        return re.sub(
            rf"({re.escape(keyword)})",
            r"<b style='color:#C0392B'>\1</b>",
            str(text), flags=re.IGNORECASE,
        )

    rows_html = ""
    for i, row in df_kw.iterrows():
        rows_html += f"""
        <tr style='border-bottom:2px solid #BDC3C7'>
          <td style='padding:6px;font-weight:bold;color:#888;vertical-align:top'>{i+1}</td>
          <td style='padding:6px;vertical-align:top'>
            <span style='font-size:0.8em;color:#aaa'>HD</span><br>
            {hl_hd(row['sentence'])}
          </td>
          <td style='padding:6px;vertical-align:top;font-family:monospace'>
            <span style='font-size:0.8em;color:#aaa'>IPA Audio (Whisper)</span><br>
            {row['ipa_audio']}
          </td>
          <td style='padding:6px;vertical-align:top;font-family:monospace;color:#777'>
            <span style='font-size:0.8em;color:#aaa'>IPA Referenz (espeak-ng)</span><br>
            {row['ipa_reference']}
          </td>
        </tr>
        """

    html = f"""
    <h3 style='margin-top:1.5em'>«{keyword}» — {len(df_kw)} Satz/Sätze</h3>
    <table style='border-collapse:collapse;width:100%;font-size:0.92em'>
      <thead>
        <tr style='background:#2C3E50;color:white'>
          <th style='padding:7px'>#</th>
          <th style='padding:7px;text-align:left'>Hochdeutsch</th>
          <th style='padding:7px;text-align:left'>IPA Audio (Whisper)</th>
          <th style='padding:7px;text-align:left'>IPA Referenz (espeak-ng)</th>
        </tr>
      </thead>
      <tbody>{rows_html}</tbody>
    </table>
    """
    display(HTML(html))
    return df_kw

## Bundesrat

Klassisches Komposita-Splitting: Ein HD-Token → 1 oder 2 IPA-Tokens, mit variierendem zweiten Teil.

In [ ]:
df_br = show_word_cases("bundesrat")

### IPA-Varianten für «bundesrat»

In [ ]:
def ipa_variants_at_position(keyword, df_kw):
    """Extrahiert IPA-Tokens an der Keyword-Position (+ optionales Folgetoken)."""
    variant_counter = Counter()
    for _, row in df_kw.iterrows():
        hd_tokens = re.sub(r"[^\w\s]", "", row["sentence"].lower()).split()
        ipa_tokens = row["ipa_audio"].strip().split()
        try:
            idx = hd_tokens.index(keyword.lower())
        except ValueError:
            continue
        tok1 = ipa_tokens[idx] if idx < len(ipa_tokens) else "—"
        tok2 = ipa_tokens[idx + 1] if idx + 1 < len(ipa_tokens) else None
        # Einfache Heuristik: tok2 gehört dazu wenn es kurz und konsonantisch beginnt
        if tok2 and len(tok2) >= 2:
            variant_counter[f"{tok1} + {tok2}"] += 1
        else:
            variant_counter[tok1] += 1

    print(f"{'IPA-Variante':<30} {'Tokens':<10} Anzahl")
    print("-" * 50)
    for variant, count in variant_counter.most_common():
        n_tok = len(variant.split(" + "))
        print(f"{variant:<30} [{n_tok} Token{'s' if n_tok > 1 else ''}]   ×{count}")
    print(f"\n→ {len(variant_counter)} verschiedene IPA-Realisierungen für 1 HD-Token.")

ipa_variants_at_position("bundesrat", df_br)

---
## Weitere Fallstudien

Hier können weitere Beispielwörter mit demselben Pattern analysiert werden.
Einfach `show_word_cases("wort")` aufrufen.

In [ ]:
# Beispiel: show_word_cases("gemeinderat")
# Beispiel: show_word_cases("stadtrat")
# Beispiel: show_word_cases("trotzdem")